# UpSet plot of Exo-MerCat's input catalog overlap
- which source catalogs (NASA/EU/OEC/TOI/EPIC) each planet appears in, stacked by status
- exports to `report/figures/exomercat_catalog_overlap_upset.pdf`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from report_plots import common


In [ ]:
from upsetplot import UpSet
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# Patch UpSet._label_sizes for NumPy 2.0+ compatibility (fixes "TypeError: only 0-dimensional arrays can be converted to Python scalars")
def _safe_float(val):
    try:
        if hasattr(val, 'ndim') and val.ndim > 0:
            return float(val[0]) if val.size > 0 else 0.0
        if hasattr(val, 'item'):
            return val.item()
        return float(val)
    except Exception:
        return val

def patched_label_sizes(self, ax, rects, where):
    if not self._show_counts and not self._show_percentages:
        return
    if self._show_counts is True:
        count_fmt = "{:.0f}"
    else:
        count_fmt = self._show_counts
        if count_fmt and "{" not in count_fmt:
            from upsetplot import util
            count_fmt = util.to_new_pos_format(count_fmt)

    pct_fmt = "{:.1%}" if self._show_percentages is True else self._show_percentages

    if count_fmt and pct_fmt:
        if where == "top":
            fmt = f"{count_fmt}\n({pct_fmt})"
        else:
            fmt = f"{count_fmt} ({pct_fmt})"

        def make_args(val):
            return _safe_float(val), _safe_float(val) / self.total
    elif count_fmt:
        fmt = count_fmt

        def make_args(val):
            return (val,)
    else:
        fmt = pct_fmt

        def make_args(val):
            return (val / self.total,)

    if where == "right":
        margin = 0.01 * abs(np.diff(ax.get_xlim()))
        for rect in rects:
            width = _safe_float(rect.get_width() + rect.get_x())
            x_coord = _safe_float(width + margin)
            y_coord = _safe_float(rect.get_y() + rect.get_height() * 0.5)
            ax.text(
                x_coord,
                y_coord,
                fmt.format(*make_args(width)),
                ha="left",
                va="center",
            )
    elif where == "left":
        margin = 0.01 * abs(np.diff(ax.get_xlim()))
        for rect in rects:
            width = _safe_float(rect.get_width() + rect.get_x())
            x_coord = _safe_float(width + margin)
            y_coord = _safe_float(rect.get_y() + rect.get_height() * 0.5)
            ax.text(
                x_coord,
                y_coord,
                fmt.format(*make_args(width)),
                ha="right",
                va="center",
            )
    elif where == "top":
        margin = 0.01 * abs(np.diff(ax.get_ylim()))
        for rect in rects:
            height = _safe_float(rect.get_height() + rect.get_y())
            x_coord = _safe_float(rect.get_x() + rect.get_width() * 0.5)
            y_coord = _safe_float(height + margin)
            ax.text(
                x_coord,
                y_coord,
                fmt.format(*make_args(height)),
                ha="center",
                va="bottom",
            )
    else:
        raise NotImplementedError("unhandled where: %r" % where)

UpSet._label_sizes = patched_label_sizes

# Patch UpSet.plot_totals to support stacked bar plotting with reordered columns
def patched_plot_totals(self, ax):
    orig_ax = ax
    ax = self._reorient(ax)
    
    by = getattr(self, '_totals_by', None)
    if by is not None:
        categories = list(self.totals.index.values)
        breakdown = []
        for cat in categories:
            df_cat = self._df[self._df.index.get_level_values(cat)]
            counts = df_cat.groupby(by).size()
            breakdown.append(counts)
            
        totals_df = pd.DataFrame(breakdown, index=categories).fillna(0)
        
        # Explicit column ordering: Confirmed on right (plotted first), Candidate on left (plotted second)
        col_order = ['Confirmed planets', 'Candidate planets']
        totals_df = totals_df.reindex(columns=[col for col in col_order if col in totals_df.columns]).fillna(0)
        
        colors = getattr(self, '_totals_colors', None)
        
        x = np.arange(len(categories))
        cum_width = None
        for col in totals_df.columns:
            width = totals_df[col]
            color = colors.get(col, self._facecolor) if isinstance(colors, dict) else self._facecolor
            rects = ax.barh(
                x,
                width,
                0.5,
                left=cum_width,
                color=color,
                align="center",
                label=col,
            )
            cum_width = width if cum_width is None else cum_width + width
        self._label_sizes(ax, rects, "left" if self._horizontal else "top")
    else:
        rects = ax.barh(
            np.arange(len(self.totals.index.values)),
            self.totals,
            0.5,
            color=self._facecolor,
            align="center",
        )
        self._label_sizes(ax, rects, "left" if self._horizontal else "top")
        for category, rect in zip(self.totals.index.values, rects):
            style = {
                k[len("bar_") :]: v
                for k, v in self.category_styles.get(category, {}).items()
                if k.startswith("bar_")
            }
            style.setdefault("edgecolor", style.get("facecolor", self._facecolor))
            for attr, val in style.items():
                getattr(rect, "set_" + attr)(val)

    max_total = getattr(self, '_totals_xlim', self.totals.max())
    if self._horizontal:
        orig_ax.set_xlim(max_total, 0)
    for x_val in ["top", "left", "right"]:
        ax.spines[self._reorient(x_val)].set_visible(False)
    ax.yaxis.set_visible(False)
    ax.xaxis.grid(True)
    ax.yaxis.grid(False)
    ax.patch.set_visible(False)

UpSet.plot_totals = patched_plot_totals

# Patch UpSet.make_grid to support gap reduction between totals and matrix subplots
def patched_make_grid(self, fig=None):
    n_cats = len(self.totals)
    n_inters = len(self.intersections)

    if fig is None:
        fig = plt.gcf()

    import matplotlib
    import warnings
    from upsetplot.plotting import RENDERER_IMPORTED
    
    try:
        from matplotlib.tight_layout import get_renderer
    except ImportError:
        try:
            from matplotlib.backend_bases import FigureCanvasBase
            get_renderer = lambda fig: fig.canvas.get_renderer()
        except Exception:
            get_renderer = None
            
    text_kw = {"size": matplotlib.rcParams["xtick.labelsize"]}
    t = fig.text(
        0,
        0,
        "\n".join(str(label) + "x" for label in self.totals.index.values),
        **text_kw,
    )
    window_extent_args = {}
    if RENDERER_IMPORTED and get_renderer is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            window_extent_args["renderer"] = get_renderer(fig)
    textw = t.get_window_extent(**window_extent_args).width
    t.remove()

    window_extent_args = {}
    if RENDERER_IMPORTED and get_renderer is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            window_extent_args["renderer"] = get_renderer(fig)
    figw = self._reorient(fig.get_window_extent(**window_extent_args)).width
    
    sizes = np.asarray([p["elements"] for p in self._subset_plots])
    fig = self._reorient(fig)
    
    non_text_nelems = len(self.intersections) + self._totals_plot_elements
    if self._element_size is None:
        colw = (figw - textw) / non_text_nelems
    else:
        render_ratio = figw / fig.get_figwidth()
        colw = self._element_size / 72 * render_ratio
        figw = colw * (non_text_nelems + np.ceil(textw / colw) + 1)
        fig.set_figwidth(figw / render_ratio)
        fig.set_figheight((colw * (n_cats + sizes.sum())) / render_ratio)

    text_nelems = int(np.ceil(figw / colw - non_text_nelems))
    
    gap_reduction = getattr(self, '_totals_gridspec_gap', 0)
    text_nelems = max(1, text_nelems - gap_reduction)
    
    # Set wspace=0 in GridSpec to ensure custom gridspec gaps work exactly as intended
    GS = self._reorient(matplotlib.gridspec.GridSpec)
    gridspec = GS(
        *self._swapaxes(
            n_cats + (sizes.sum() or 0),
            n_inters + text_nelems + self._totals_plot_elements,
        ),
        hspace=1,
        wspace=0,
    )
    if self._horizontal:
        out = {
            "matrix": gridspec[-n_cats:, -n_inters:],
            "shading": gridspec[-n_cats:, :],
            "totals": None
            if self._totals_plot_elements == 0
            else gridspec[-n_cats:, : self._totals_plot_elements],
            "gs": gridspec,
        }
        cumsizes = np.cumsum(sizes[::-1])
        for start, stop, plot in zip(
            np.hstack([[0], cumsizes]), cumsizes, self._subset_plots[::-1]
        ):
            out[plot["id"]] = gridspec[start:stop, -n_inters:]
    else:
        out = {
            "matrix": gridspec[-n_inters:, :n_cats],
            "shading": gridspec[:, :n_cats],
            "totals": None
            if self._totals_plot_elements == 0
            else gridspec[: self._totals_plot_elements, :n_cats],
            "gs": gridspec,
        }
        cumsizes = np.cumsum(sizes)
        for start, stop, plot in zip(
            np.hstack([[0], cumsizes]), cumsizes, self._subset_plots
        ):
            out[plot["id"]] = gridspec[-n_inters:, start + n_cats : stop + n_cats]
    return out

UpSet.make_grid = patched_make_grid

mpl.rcParams['figure.dpi'] = 300


In [ ]:
df = pd.read_csv(common.ROOT / 'input' / 'exo-mercat.csv', low_memory=False)
df = df[df["status"] != "FALSE POSITIVE"]
# catalogs = ['eu', 'nasa', 'oec', 'toi', 'epic']
catalogs = ['nasa', 'eu', 'oec', 'toi', 'epic']

for cat in catalogs:
    df[cat.upper()] = df['catalog'].str.contains(cat, na=False)
    
# Differentiate between confirmed and candidate planets
df['status_group'] = np.where(df['status'] == 'CONFIRMED', 'Confirmed planets', 'Candidate planets')

df = df.set_index([cat.upper() for cat in catalogs])

In [ ]:
with plt.rc_context({'font.size': 13}):
    fig = plt.figure(figsize=(6, 4))

    upset = UpSet(
        df,
        show_counts=True,
        sort_by='degree',
        sort_categories_by='input',
        element_size=35,
        min_subset_size=10,
        intersection_plot_elements=0,
        totals_plot_elements=10,
    )

    # Enable stacked bars for intersections and make it taller by setting elements=6
    colors_map = {'Confirmed planets': '#555555', 'Candidate planets': '#cbcbcb'}
    upset.add_stacked_bars(
        by='status_group', 
        colors=colors_map, 
        elements=10
    )

    # Enable stacked bars for totals
    upset._totals_by = 'status_group'
    upset._totals_colors = colors_map
    upset._totals_xlim = 10000

    axes = upset.plot(fig)

t = """NASA = Nasa Exoplanet Archive
EU   = Extrasolar Planets Encyclopaedia
OEC  = Open Exoplanet Catalogue
EPIC = EPIC/K2 Objects of Interest Catalog
TOI  = TESS Object of Interest"""


axes["extra0"].text(
    -0.65,0.35, 
    t,
    c="black",
    transform=axes["extra0"].transAxes,
    ha="left", 
    va="top", 
    fontsize=12,
    linespacing=1.5,
    bbox=dict(boxstyle="round,pad=0.7", facecolor="white", edgecolor="black") )
# draw a box around the text

# plt.show()
common.save_figure("exomercat_catalog_overlap_upset")